In [ ]:
#  import python libariries 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#  import models  from Sk-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB



In [ ]:
'''
data= pd.read_csv("student_math.csv")
 data.isnull().sum() # not null values  
# convrt into classification problem 
 data["Result"] = data["G3"].apply(lambda x: 1 if x >= 10 else 0)
 data.head()
 data = data.drop("G3", axis=1)
'''

In [ ]:
#  load the dataset
df= pd.read_csv('dataset_math.csv')
#  data does not have null values
df = df.drop("Unnamed: 0", axis=1)


In [ ]:
# categorical columns and numerical columns 
numerical_columns=df.select_dtypes(include =['number']).columns 
categorical_columns= df.select_dtypes(include=['object']).columns

In [ ]:
#  data that does not contain missing vaues 
classes_count= df["Result"].value_counts()
df["Result"] = df["Result"].astype(int)
# Result
# 1.0    265  student pass 
# 0.0    130 student fails


In [ ]:
# perform eda 
plt.pie(
    classes_count,
    labels=['Yes','No'],
    autopct='%1.1f%%',
)
plt.title("Students passed or failed")
#  67.1 -> passed and 32.9 fails


In [ ]:
# Numerical columns

numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

fig, axs = plt.subplots(4, 4, figsize=(18, 12))
axes = axs.flatten()

for ax, col in zip(axes, numeric_columns):
    sns.histplot(
        data=df,
        x=col,
        bins=10,
        kde=True,
        ax=ax,
        color='skyblue'
    )

    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Frequency")

# Remove unused subplots (if any)
for i in range(len(numeric_columns), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig("02_numeric_histograms.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# analyze the category data 

# sex_count= df['sex'].value_counts()
# sex_bar=sns.barplot(sex_count)
# sex_bar.bar_label(sex_bar.containers[0])


cat_columns=['school',  'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic']
fig, axs = plt.subplots(4, 4, figsize=(18, 10))
axes = axs.flatten() # convert 2d array into 1d for looping
for ax, col in zip(axes, cat_columns):
    count = df[col].value_counts()
    bars = sns.barplot(
        x=count.index,
        y=count.values,
        ax=ax
    )
    bars.bar_label(bars.containers[0])
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("01_categ_features.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Analyze numerical data using box plots

fig, axs = plt.subplots(4, 4, figsize=(18, 10))
axes = axs.flatten()

for ax, col in zip(axes, numeric_columns):
    box = sns.boxplot(
        y=df[col],
        ax=ax
    )

    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Value")

plt.tight_layout()

plt.savefig("03_numeric_features_outliers.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

In [ ]:
# Find the number of outliers using IQR
for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print(f"{col}: {len(outliers)} outliers")
    

In [ ]:
# Find outliers in Absences using the IQR method

Q1 = df['absences'].quantile(0.25)
Q3 = df['absences'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Extract the outlier students
absence_outliers = df[
    (df['absences'] < lower_bound) |
    (df['absences'] > upper_bound)
]

# Create a small table
outlier_table = absence_outliers[
    ['absences', 'G1', 'G2', 'Result']
].sort_values('absences')

display(outlier_table)



In [61]:

# Encoding Part

# Separate features and target
X = df.drop(columns=['Result'])
Y = df['Result']
# Result is the target variable
numeric_features = [
    col for col in numerical_columns
    if col != 'Result'
]

# Import preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',
         OneHotEncoder(handle_unknown='ignore'),
         categorical_columns),

        ('num',
         StandardScaler(),
         numeric_features)
    ]
)


# Apply encoding and scaling
X_encoded = preprocessor.fit_transform(X)


# Get encoded feature names
feature_names = preprocessor.get_feature_names_out()


# Convert encoded data into DataFrame
X_encoded_df = pd.DataFrame(
    X_encoded.toarray() if hasattr(X_encoded, "toarray") else X_encoded,
    columns=feature_names
)


# Add Result column back
X_encoded_df['Result'] = Y.values


# Save encoded dataset
X_encoded_df.to_csv(
    'encoded.csv',
    index=False
)


print("\nEncoding completed successfully!")
print("Encoded dataset shape:", X_encoded_df.shape)
print("\nEncoded dataset:")
print(X_encoded_df.head())


Encoding completed successfully!
Encoded dataset shape: (395, 59)

Encoded dataset:
   cat__school_GP  cat__school_MS  cat__sex_F  cat__sex_M  cat__address_R  \
0             1.0             0.0         1.0         0.0             0.0   
1             1.0             0.0         1.0         0.0             0.0   
2             1.0             0.0         1.0         0.0             0.0   
3             1.0             0.0         1.0         0.0             0.0   
4             1.0             0.0         1.0         0.0             0.0   

   cat__address_U  cat__famsize_GT3  cat__famsize_LE3  cat__Pstatus_A  \
0             1.0               1.0               0.0             1.0   
1             1.0               1.0               0.0             0.0   
2             1.0               0.0               1.0             0.0   
3             1.0               1.0               0.0             0.0   
4             1.0               1.0               0.0             0.0   

   cat__Pstat

In [67]:
X_encoded_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 59 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cat__school_GP          395 non-null    float64
 1   cat__school_MS          395 non-null    float64
 2   cat__sex_F              395 non-null    float64
 3   cat__sex_M              395 non-null    float64
 4   cat__address_R          395 non-null    float64
 5   cat__address_U          395 non-null    float64
 6   cat__famsize_GT3        395 non-null    float64
 7   cat__famsize_LE3        395 non-null    float64
 8   cat__Pstatus_A          395 non-null    float64
 9   cat__Pstatus_T          395 non-null    float64
 10  cat__Mjob_at_home       395 non-null    float64
 11  cat__Mjob_health        395 non-null    float64
 12  cat__Mjob_other         395 non-null    float64
 13  cat__Mjob_services      395 non-null    float64
 14  cat__Mjob_teacher       395 non-null    fl

Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2',
       'Result'],
      dtype='object')